
# Part 1: Starting from High Explainability

Decision Trees and GAM's might be a natural starting place for models where explainability could be critical. Insight into exactly what is driving the classifier could drive your advertising spend in future campaigns or even as this one continues. 


In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from torch.utils.data import DataLoader, TensorDataset
import time
from datetime import datetime
import numpy as np
import pandas as pd
import torch 
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score

In [ ]:


marketing_file = "./dataset/marketing_campaign.csv"

marketing_df = pd.read_csv("./dataset/marketing_campaign.csv", sep='\t', encoding="ascii")


In [ ]:
marketing_df.info()

In [ ]:
marketing_df.describe()

In [ ]:
# Fill null values with median
marketing_df['Income'] = marketing_df['Income'].fillna(marketing_df['Income'].median())

In [ ]:
# Dt_Customer ->datetime
marketing_df['Dt_Customer'] = pd.to_datetime(marketing_df['Dt_Customer'], format='%d-%m-%Y')

reference_date = datetime(2014, 7, 1)
marketing_df['Customer_Since_Months'] = (reference_date - marketing_df['Dt_Customer']).dt.days // 30

marketing_df.drop(columns=['Dt_Customer'], inplace=True)

In [ ]:
print(marketing_df['Education'].value_counts())
marketing_df = pd.get_dummies(marketing_df, columns=['Education'], drop_first=False)

In [ ]:
marketing_df['Marital_Status'] = marketing_df['Marital_Status'].replace({'Absurd': 'Other', 'YOLO': 'Other'})

print(marketing_df['Marital_Status'].value_counts())
marketing_df = pd.get_dummies(marketing_df, columns=['Marital_Status'], drop_first=False)

In [ ]:
marketing_df['Age'] = 2015 - marketing_df['Year_Birth']

marketing_df.drop(columns=['Year_Birth'], inplace=True)


In [ ]:
irrelevant_columns = ['ID', 'NumDealsPurchases', 'Response']
marketing_df = marketing_df.drop(columns=irrelevant_columns)

In [ ]:
marketing_df['AcceptedAny'] = (
    (marketing_df['AcceptedCmp1'] == 1) | 
    (marketing_df['AcceptedCmp2'] == 1) | 
    (marketing_df['AcceptedCmp3'] == 1) | 
    (marketing_df['AcceptedCmp4'] == 1) | 
    (marketing_df['AcceptedCmp5'] == 1)
).astype(int)

marketing_df.drop(columns=['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5'], inplace=True)

print(marketing_df[['AcceptedAny']].value_counts())

In [ ]:
marketing_df['Education'] = marketing_df[['Education_Basic', 'Education_2n Cycle', 'Education_Graduation', 'Education_Master', 'Education_PhD']].idxmax(axis=1)

marketing_df['Marital_Status'] = marketing_df[['Marital_Status_Single', 'Marital_Status_Together', 'Marital_Status_Married', 
                                               'Marital_Status_Divorced', 'Marital_Status_Widow', 'Marital_Status_Alone',
                                               'Marital_Status_Other']].idxmax(axis=1)
marketing_df.drop(columns=['Education_Basic', 'Education_2n Cycle', 'Education_Graduation', 'Education_Master', 'Education_PhD',
                           'Marital_Status_Single', 'Marital_Status_Together', 'Marital_Status_Married', 'Marital_Status_Divorced', 
                           'Marital_Status_Widow', 'Marital_Status_Alone', 'Marital_Status_Other'], inplace=True)

print(marketing_df[['Education', 'Marital_Status']].head())

In [ ]:
marketing_df['Education'] = marketing_df['Education'].str.replace('Education_', '')
marketing_df['Marital_Status'] = marketing_df['Marital_Status'].str.replace('Marital_Status_', '')

print(marketing_df[['Education', 'Marital_Status']].head())

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_enc = LabelEncoder()
marketing_df['Education'] = label_enc.fit_transform(marketing_df['Education'])
marketing_df['Marital_Status'] = label_enc.fit_transform(marketing_df['Marital_Status'])

print(marketing_df[['Education', 'Marital_Status']].value_counts())

In [ ]:
def split_data(marketing_df, method='train_val_test', k=5, val_size=0.2, test_size=0.2):
    
    X = marketing_df.drop(columns=['AcceptedAny']).values
    y = marketing_df['AcceptedAny'].values

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    if method == 'train_val_test':
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )
        
        val_size_adjusted = val_size / (1 - test_size) 
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_val, y_train_val, test_size=val_size_adjusted, random_state=42, stratify=y_train_val
        )

        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
        X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
        y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
        y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

        print(f"Train: {X_train_tensor.shape[0]}, Val: {X_val_tensor.shape[0]}, Test: {X_test_tensor.shape[0]}")
        return X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor, X_test_tensor, y_test_tensor

    elif method == 'cross_val':
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
        folds = []

        for train_index, test_index in skf.split(X, y):
            X_train_fold, X_test_fold = X[train_index], X[test_index]
            y_train_fold, y_test_fold = y[train_index], y[test_index]

            X_train_tensor = torch.tensor(X_train_fold, dtype=torch.float32)
            y_train_tensor = torch.tensor(y_train_fold, dtype=torch.float32).unsqueeze(1)
            X_test_tensor = torch.tensor(X_test_fold, dtype=torch.float32)
            y_test_tensor = torch.tensor(y_test_fold, dtype=torch.float32).unsqueeze(1)

            folds.append((X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor))

        print(f"Train: {folds[0][0].shape[0]}, Test: {folds[0][2].shape[0]}")
        return folds


In [ ]:

# Drop constant columns that cause NaN when StandardScaler divides by std=0
# Z_CostContact is always 3, Z_Revenue is always 11 — they carry no information
constant_cols = [col for col in marketing_df.columns if marketing_df[col].nunique() <= 1]
print(f"Dropping constant columns: {constant_cols}")
marketing_df.drop(columns=constant_cols, inplace=True)

In [ ]:
# ============================================================
# Decision Tree
# ============================================================

from sklearn.tree import DecisionTreeClassifier

feature_names = marketing_df.drop(columns=['AcceptedAny']).columns.tolist()

X_train, y_train, X_val, y_val, X_test, y_test = split_data(marketing_df, method='train_val_test')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# --- Hyperparameter tuning: max_depth via GridSearchCV ---
param_grid = {'max_depth': [5, 7, 10, 15, 20, None]}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

start_time = time.time()
grid_search.fit(X_train_scaled, y_train.ravel())
end_time = time.time()

best_depth = grid_search.best_params_['max_depth']
best_dt_cv_f1 = grid_search.best_score_
print(f"Best Max Depth: {best_depth}, CV F1: {best_dt_cv_f1:.4f}")
print(f"Grid search time: {end_time - start_time:.4f}s")

print("\n{:<12} {:<10}".format("Max Depth", "CV F1"))
for mean, params in zip(grid_search.cv_results_['mean_test_score'],
                        grid_search.cv_results_['params']):
    print("{:<12} {:<10.4f}".format(str(params['max_depth']), mean))

In [ ]:
# --- Evaluate best DT on test set ---
dt_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)

start_time = time.time()
dt_model.fit(X_train_scaled, y_train.ravel())
end_time = time.time()
dt_training_time = end_time - start_time

y_test_pred = dt_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)

print(f"\nDecision Tree (max_depth={best_depth}) Test Results:")
print(f"Accuracy: {accuracy:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"Training Time: {dt_training_time:.4f} seconds")

In [ ]:

# --- Train size vs error ---
train_sizes = [400, 800, 1344]
dt_size_results = []

for size in train_sizes:
    X_train_subset = X_train_scaled[:size]
    y_train_subset = y_train[:size]

    dt_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
    dt_model.fit(X_train_subset, y_train_subset.ravel())

    y_train_pred = dt_model.predict(X_train_subset)
    train_error = 1 - accuracy_score(y_train_subset, y_train_pred)

    y_test_pred = dt_model.predict(X_test_scaled)
    test_error = 1 - accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)

    dt_size_results.append({'train_size': size, 'train_error': train_error, 'test_error': test_error, 'test_f1': test_f1})
    print(f"Train Size: {size} | Train Error: {train_error:.4f} | Test Error: {test_error:.4f} | Test F1: {test_f1:.4f}")

dt_size_results_arr = np.array([(r['train_size'], r['train_error'], r['test_error']) for r in dt_size_results])

plt.figure(figsize=(10, 5))
plt.title("Decision Tree: Train Size vs Training and Testing Error")
plt.xlabel("Training Size")
plt.ylabel("Error")
plt.plot(dt_size_results_arr[:, 0], dt_size_results_arr[:, 1], marker='o', linestyle='-', label="Training Error")
plt.plot(dt_size_results_arr[:, 0], dt_size_results_arr[:, 2], marker='s', linestyle='-', label="Testing Error")
plt.legend()
plt.grid(True)
plt.savefig("dt_train_size_vs_error.png")
plt.show()

In [ ]:
# --- Train size vs F1 ---
plt.figure(figsize=(10, 5))
plt.title("Decision Tree: Train Size vs Test F1-Score")
plt.xlabel("Training Size")
plt.ylabel("F1-Score")
plt.plot([r['train_size'] for r in dt_size_results], [r['test_f1'] for r in dt_size_results], marker='o', label="Test F1-Score")
plt.legend()
plt.grid(True)
plt.savefig("dt_train_size_vs_f1.png")
plt.show()

In [ ]:
# --- Feature Importance ---
importances = dt_model.feature_importances_
indices = np.argsort(importances)[::-1]
sorted_features = [feature_names[i] for i in indices]
sorted_importances = importances[indices]

plt.figure(figsize=(10, 6))
plt.title("Decision Tree: Feature Importances")
plt.barh(range(len(sorted_features)), sorted_importances[::-1])
plt.yticks(range(len(sorted_features)), sorted_features[::-1])
plt.xlabel("Importance (Gini)")
plt.tight_layout()
plt.savefig("dt_feature_importance.png")
plt.show()

print("\nTop 10 features by importance:")
for feat, imp in zip(sorted_features[:10], sorted_importances[:10]):
    print(f"  {feat:<30} {imp:.4f}")

In [ ]:
# --- Tree Visualization (first 3 levels for readability) ---
from sklearn.tree import plot_tree

plt.figure(figsize=(24, 10))
plt.title(f"Decision Tree Structure (max_depth={best_depth}, showing first 3 levels)")
plot_tree(
    dt_model,
    feature_names=feature_names,
    class_names=['Not Accepted', 'Accepted'],
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=3
)
plt.tight_layout()
plt.savefig("dt_tree_visualization.png", dpi=150)
plt.show()

In [ ]:
# --- Level-by-Level Analysis ---
from sklearn.tree import _tree
from collections import defaultdict, deque

tree_ = dt_model.tree_
node_feature = [
    feature_names[i] if i != _tree.TREE_UNDEFINED else None
    for i in tree_.feature
]

# BFS to group nodes by depth
levels = defaultdict(list)
queue = deque([(0, 0)])
while queue:
    node_id, depth = queue.popleft()
    levels[depth].append(node_id)
    left = tree_.children_left[node_id]
    right = tree_.children_right[node_id]
    if left != _tree.TREE_LEAF:
        queue.append((left, depth + 1))
        queue.append((right, depth + 1))

print(f"Tree depth: {dt_model.get_depth()} | Total nodes: {tree_.node_count}\n")
print(f"{'Level':<7} {'Nodes':<7} {'Splits':<8} {'Leaves':<8} {'Samples covered':<18} {'Features used at this level'}")
print("-" * 90)

for depth in sorted(levels.keys()):
    nodes = levels[depth]
    split_nodes = [n for n in nodes if tree_.children_left[n] != _tree.TREE_LEAF]
    leaf_nodes  = [n for n in nodes if tree_.children_left[n] == _tree.TREE_LEAF]
    total_samples = sum(tree_.n_node_samples[n] for n in nodes)

    feat_counts = defaultdict(int)
    for n in split_nodes:
        feat_counts[node_feature[n]] += 1
    feat_str = ", ".join(f"{f}×{c}" if c > 1 else f for f, c in
                         sorted(feat_counts.items(), key=lambda x: -x[1]))

    print(f"{depth:<7} {len(nodes):<7} {len(split_nodes):<8} {len(leaf_nodes):<8} {total_samples:<18} {feat_str}")

print("\n--- Split details by level ---")
for depth in sorted(levels.keys()):
    split_nodes = [n for n in levels[depth] if tree_.children_left[n] != _tree.TREE_LEAF]
    if not split_nodes:
        continue
    print(f"\nLevel {depth}:")
    for n in split_nodes:
        feat   = node_feature[n]
        thresh = tree_.threshold[n]
        samples = tree_.n_node_samples[n]
        values = tree_.value[n][0]
        majority = "Accepted" if values[1] > values[0] else "Not Accepted"
        print(f"  Node {n:>4}: {feat} <= {thresh:.3f}  "
              f"| samples={samples}  | class dist=[{int(values[0])}, {int(values[1])}]  | majority={majority}")

In [ ]:
# ============================================================
# GAM (Generalized Additive Model)
# ============================================================

from pygam import LogisticGAM

X_train, y_train, X_val, y_val, X_test, y_test = split_data(marketing_df, method='train_val_test')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# --- Hyperparameter tuning: n_splines and lam via pygam gridsearch ---
# pygam doesn't implement the sklearn estimator interface, so we use its
# built-in .gridsearch() to tune lam, and loop over n_splines ourselves.
n_splines_options = [5, 10, 20]
lam_values = np.logspace(-1, 1, 3)  # [0.1, 1.0, 10.0]

gam_results = []
for n in n_splines_options:
    candidate = LogisticGAM(n_splines=n)

    start_time = time.time()
    candidate.gridsearch(X_train_scaled, y_train.ravel(), lam=lam_values)
    elapsed = time.time() - start_time

    y_val_pred = candidate.predict(X_val_scaled)
    val_f1 = f1_score(y_val, y_val_pred)

    # .lam is a nested list [[val], [val], ...] after gridsearch — take the first term
    best_lam = candidate.lam[0][0]
    gam_results.append({'n_splines': n, 'lam': best_lam, 'val_f1': val_f1, 'training_time': elapsed})

print("{:<12} {:<10} {:<10} {:<15}".format("n_splines", "lam", "F1-Score", "Training Time (s)"))
for r in gam_results:
    print("{:<12} {:<10} {:<10.4f} {:<15.4f}".format(r['n_splines'], r['lam'], r['val_f1'], r['training_time']))

best_gam = max(gam_results, key=lambda x: x['val_f1'])
print(f"\nBest Config: n_splines={best_gam['n_splines']}, lam={best_gam['lam']}, Val F1: {best_gam['val_f1']:.4f}")

In [ ]:
# --- Evaluate best GAM on test set ---
gam_model = LogisticGAM(n_splines=best_gam['n_splines'], lam=best_gam['lam'])

start_time = time.time()
gam_model.fit(X_train_scaled, y_train.ravel())
end_time = time.time()
gam_training_time = end_time - start_time

y_test_pred = gam_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)

print(f"\nGAM (n_splines={best_gam['n_splines']}, lam={best_gam['lam']}) Test Results:")
print(f"Accuracy: {accuracy:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"Training Time: {gam_training_time:.4f} seconds")

In [ ]:
# --- Train size vs error ---
train_sizes = [400, 800, 1344]
gam_size_results = []

for size in train_sizes:
    X_train_subset = X_train_scaled[:size]
    y_train_subset = y_train[:size]

    gam_model = LogisticGAM(n_splines=best_gam['n_splines'], lam=best_gam['lam'])
    gam_model.fit(X_train_subset, y_train_subset.ravel())

    y_train_pred = gam_model.predict(X_train_subset)
    train_error = 1 - accuracy_score(y_train_subset, y_train_pred)

    y_test_pred = gam_model.predict(X_test_scaled)
    test_error = 1 - accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)

    gam_size_results.append({'train_size': size, 'train_error': train_error, 'test_error': test_error, 'test_f1': test_f1})
    print(f"Train Size: {size} | Train Error: {train_error:.4f} | Test Error: {test_error:.4f} | Test F1: {test_f1:.4f}")

gam_size_results_arr = np.array([(r['train_size'], r['train_error'], r['test_error']) for r in gam_size_results])

plt.figure(figsize=(10, 5))
plt.title("GAM: Train Size vs Training and Testing Error")
plt.xlabel("Training Size")
plt.ylabel("Error")
plt.plot(gam_size_results_arr[:, 0], gam_size_results_arr[:, 1], marker='o', linestyle='-', label="Training Error")
plt.plot(gam_size_results_arr[:, 0], gam_size_results_arr[:, 2], marker='s', linestyle='-', label="Testing Error")
plt.legend()
plt.grid(True)
plt.savefig("gam_train_size_vs_error.png")
plt.show()

In [ ]:
# --- Train size vs F1 ---
plt.figure(figsize=(10, 5))
plt.title("GAM: Train Size vs Test F1-Score")
plt.xlabel("Training Size")
plt.ylabel("F1-Score")
plt.plot([r['train_size'] for r in gam_size_results], [r['test_f1'] for r in gam_size_results], marker='o', label="Test F1-Score")
plt.legend()
plt.grid(True)
plt.savefig("gam_train_size_vs_f1.png")
plt.show()

In [ ]:
# --- Per-feature shape functions ---
# A GAM decomposes predictions as f(X) = β₀ + f₁(x₁) + f₂(x₂) + ... + fₙ(xₙ).
# Each shape function shows exactly how that feature shifts the log-odds of acceptance.

n_features = len(feature_names)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, feat in enumerate(feature_names):
    XX = gam_model.generate_X_grid(term=i)
    pdep, confi = gam_model.partial_dependence(term=i, X=XX, width=0.95)

    axes[i].plot(XX[:, i], pdep, color='steelblue')
    axes[i].fill_between(XX[:, i], confi[:, 0], confi[:, 1], alpha=0.3, color='steelblue')
    axes[i].axhline(0, color='gray', linestyle='--', linewidth=0.8)
    axes[i].set_title(feat)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel("Partial effect (log-odds)")
    axes[i].grid(True, alpha=0.3)

for j in range(n_features, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("GAM: Per-Feature Shape Functions\n(effect on log-odds of campaign acceptance)", y=1.02)
plt.tight_layout()
plt.savefig("gam_shape_functions.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# XGBoost (Gradient Boosted Decision Tree)
# ============================================================

from xgboost import XGBClassifier

X_train, y_train, X_val, y_val, X_test, y_test = split_data(marketing_df, method='train_val_test')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# --- Hyperparameter tuning via GridSearchCV ---
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.1],
}

grid_search = GridSearchCV(
    XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

start_time = time.time()
grid_search.fit(X_train_scaled, y_train.ravel())
end_time = time.time()

best_xgb = grid_search.best_params_
best_xgb_cv_f1 = grid_search.best_score_
print(f"Best Config: {best_xgb}, CV F1: {best_xgb_cv_f1:.4f}")
print(f"Grid search time: {end_time - start_time:.4f}s")

print("\n{:<15} {:<12} {:<15} {:<10}".format("n_estimators", "max_depth", "learning_rate", "CV F1"))
for mean, params in zip(grid_search.cv_results_['mean_test_score'],
                        grid_search.cv_results_['params']):
    print("{:<15} {:<12} {:<15} {:<10.4f}".format(
        params['n_estimators'], params['max_depth'], params['learning_rate'], mean))

In [ ]:

# --- Evaluate best XGBoost on test set ---
xgb_model = XGBClassifier(
    n_estimators=best_xgb['n_estimators'],
    max_depth=best_xgb['max_depth'],
    learning_rate=best_xgb['learning_rate'],
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

start_time = time.time()
xgb_model.fit(X_train_scaled, y_train.ravel())
end_time = time.time()
xgb_training_time = end_time - start_time

y_test_pred = xgb_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)

print(f"\nXGBoost Test Results:")
print(f"Accuracy: {accuracy:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"Training Time: {xgb_training_time:.4f} seconds")

In [ ]:

# --- Train size vs error ---
train_sizes = [400, 800, 1344]
xgb_size_results = []

for size in train_sizes:
    X_train_subset = X_train_scaled[:size]
    y_train_subset = y_train[:size]

    xgb_model = XGBClassifier(
        n_estimators=best_xgb['n_estimators'],
        max_depth=best_xgb['max_depth'],
        learning_rate=best_xgb['learning_rate'],
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
    xgb_model.fit(X_train_subset, y_train_subset.ravel())

    y_train_pred = xgb_model.predict(X_train_subset)
    train_error = 1 - accuracy_score(y_train_subset, y_train_pred)

    y_test_pred = xgb_model.predict(X_test_scaled)
    test_error = 1 - accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)

    xgb_size_results.append({'train_size': size, 'train_error': train_error, 'test_error': test_error, 'test_f1': test_f1})
    print(f"Train Size: {size} | Train Error: {train_error:.4f} | Test Error: {test_error:.4f} | Test F1: {test_f1:.4f}")

xgb_size_results_arr = np.array([(r['train_size'], r['train_error'], r['test_error']) for r in xgb_size_results])

plt.figure(figsize=(10, 5))
plt.title("XGBoost: Train Size vs Training and Testing Error")
plt.xlabel("Training Size")
plt.ylabel("Error")
plt.plot(xgb_size_results_arr[:, 0], xgb_size_results_arr[:, 1], marker='o', linestyle='-', label="Training Error")
plt.plot(xgb_size_results_arr[:, 0], xgb_size_results_arr[:, 2], marker='s', linestyle='-', label="Testing Error")
plt.legend()
plt.grid(True)
plt.savefig("xgb_train_size_vs_error.png")
plt.show()

In [ ]:

# --- Train size vs F1 ---
plt.figure(figsize=(10, 5))
plt.title("XGBoost: Train Size vs Test F1-Score")
plt.xlabel("Training Size")
plt.ylabel("F1-Score")
plt.plot([r['train_size'] for r in xgb_size_results], [r['test_f1'] for r in xgb_size_results], marker='o', label="Test F1-Score")
plt.legend()
plt.grid(True)
plt.savefig("xgb_train_size_vs_f1.png")
plt.show()